In [1]:
# ============================================
# STEP 1: LOAD FINAL RANDOM FOREST MODEL
# ============================================

import os
import joblib

# Path to the saved final model
model_path = "../models/random_forest_final.pkl"

# Check if model file exists
print("Model exists:", os.path.exists(model_path))

# Load the trained Random Forest model
model = joblib.load(model_path)

print("Final Random Forest model loaded successfully!")
print("Model type:", type(model))

Model exists: True
Final Random Forest model loaded successfully!
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [2]:
# ============================================
# STEP 2: LOAD FEATURE DATASET
# ============================================

import pandas as pd
import numpy as np

# Load extracted features
features_df = pd.read_csv("../data/features.csv")

print("Feature Dataset Loaded Successfully!")

print("\nDataset Shape:")
print(features_df.shape)

print("\nFeature Columns:")
print(features_df.columns.tolist())

print("\nFirst 5 Rows:")
display(features_df.head())

Feature Dataset Loaded Successfully!

Dataset Shape:
(13181, 9)

Feature Columns:
['Mean', 'Std', 'Variance', 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma', 'Label']

First 5 Rows:


,Mean,Std,Variance,Delta,Theta,Alpha,Beta,Gamma,Label
0,6.224838e-06,0.000040,2.019072e-09,5.086776e-10,1.252318e-10,4.501589e-11,9.936528e-11,5.169968e-11,0
1,-6.238922e-07,0.000035,1.604471e-09,1.510739e-09,1.018136e-10,3.810161e-11,1.113378e-10,7.269863e-11,0
2,9.433753e-07,0.000047,3.111549e-09,2.225362e-09,2.902292e-10,5.649391e-11,1.265199e-10,9.276142e-11,0
3,2.200194e-07,0.000031,1.184050e-09,6.634073e-10,1.910103e-10,4.744904e-11,2.174806e-10,1.205579e-10,0
4,-1.851659e-07,0.000025,7.406582e-10,3.636297e-10,9.952621e-11,3.020521e-11,1.349834e-10,7.228049e-11,0


In [3]:
# ============================================
# STEP 3: PREPARE ONE SAMPLE FOR PREDICTION
# ============================================

# Feature columns used during model training
feature_columns = [
    "Mean",
    "Std",
    "Variance",
    "Delta",
    "Theta",
    "Alpha",
    "Beta",
    "Gamma"
]

# Select one sample from the dataset
sample_index = 0

# Extract only the feature values
sample_features = features_df.loc[
    sample_index,
    feature_columns
].values.reshape(1, -1)

# Get the true label for comparison
true_label = features_df.loc[
    sample_index,
    "Label"
]

print("Sample prepared successfully!")

print("\nSample Index:", sample_index)
print("Feature Shape:", sample_features.shape)
print("True Label:", true_label)

print("\nFeature Values:")
print(sample_features)

Sample prepared successfully!

Sample Index: 0
Feature Shape: (1, 8)
True Label: 0

Feature Values:
[[6.22483826e-06 3.98636602e-05 2.01907206e-09 5.08677551e-10
  1.25231785e-10 4.50158945e-11 9.93652783e-11 5.16996820e-11]]


In [4]:
# ============================================
# STEP 4: PREDICT EEG SAMPLE
# ============================================

# Final operating threshold selected during evaluation
final_threshold = 0.40

# Get seizure probability
seizure_probability = model.predict_proba(sample_features)[0, 1]

# Apply final threshold
predicted_label = int(seizure_probability >= final_threshold)

# Convert label to class name
if predicted_label == 1:
    predicted_class = "Seizure"
else:
    predicted_class = "Normal"

# Convert true label to class name
if true_label == 1:
    true_class = "Seizure"
else:
    true_class = "Normal"

# Display results
print("==========================================")
print("EEG SEIZURE PREDICTION")
print("==========================================")

print(f"Seizure Probability: {seizure_probability:.4f}")
print(f"Final Threshold:     {final_threshold:.2f}")

print(f"\nPredicted Class: {predicted_class}")
print(f"True Class:      {true_class}")

print("\nPrediction Correct:",
      predicted_label == true_label)

EEG SEIZURE PREDICTION
Seizure Probability: 0.0000
Final Threshold:     0.40

Predicted Class: Normal
True Class:      Normal

Prediction Correct: True


In [5]:
# ============================================
# STEP 5: TEST A SEIZURE SAMPLE
# ============================================

# Find indices of seizure samples
seizure_indices = features_df.index[
    features_df["Label"] == 1
].tolist()

# Select the first seizure sample
seizure_sample_index = seizure_indices[0]

# Extract its 8 features
seizure_sample_features = features_df.loc[
    seizure_sample_index,
    feature_columns
].values.reshape(1, -1)

# Get true label
seizure_true_label = features_df.loc[
    seizure_sample_index,
    "Label"
]

# Predict seizure probability
seizure_probability = model.predict_proba(
    seizure_sample_features
)[0, 1]

# Apply final threshold
seizure_predicted_label = int(
    seizure_probability >= final_threshold
)

# Convert labels to names
predicted_class = (
    "Seizure" if seizure_predicted_label == 1
    else "Normal"
)

true_class = (
    "Seizure" if seizure_true_label == 1
    else "Normal"
)

# Display result
print("==========================================")
print("SEIZURE SAMPLE PREDICTION")
print("==========================================")

print("Sample Index:", seizure_sample_index)
print(f"Seizure Probability: {seizure_probability:.4f}")
print(f"Final Threshold:     {final_threshold:.2f}")

print(f"\nPredicted Class: {predicted_class}")
print(f"True Class:      {true_class}")

print("\nPrediction Correct:",
      seizure_predicted_label == seizure_true_label)

SEIZURE SAMPLE PREDICTION
Sample Index: 1649
Seizure Probability: 0.6100
Final Threshold:     0.40

Predicted Class: Seizure
True Class:      Seizure

Prediction Correct: True


In [6]:
# ============================================
# STEP 6: FINAL PREDICTION SUMMARY
# ============================================

print("==========================================")
print("FINAL EEG PREDICTION PIPELINE SUMMARY")
print("==========================================")

print("\nModel:")
print("Random Forest Classifier")

print("\nOperating Threshold:")
print(final_threshold)

print("\nNormal Sample:")
print("Predicted: Normal")
print("True: Normal")
print("Correct: True")

print("\nSeizure Sample:")
print("Sample Index:", seizure_sample_index)
print(f"Seizure Probability: {seizure_probability:.4f}")
print("Predicted: Seizure")
print("True: Seizure")
print("Correct: True")

print("\n==========================================")
print("Prediction Pipeline Completed Successfully!")
print("==========================================")

FINAL EEG PREDICTION PIPELINE SUMMARY

Model:
Random Forest Classifier

Operating Threshold:
0.4

Normal Sample:
Predicted: Normal
True: Normal
Correct: True

Seizure Sample:
Sample Index: 1649
Seizure Probability: 0.6100
Predicted: Seizure
True: Seizure
Correct: True

Prediction Pipeline Completed Successfully!
